In [1]:
import archs4py as a4
import pandas as pd
import numpy as np

#path to file
file = "./data/human_tpm_v2.latest.h5"

In [2]:
meta_meta = a4.meta.meta(file, "liver", meta_fields=["title", "characteristics_ch1", "source_name_ch1", 'library_source', 'library_strategy', 'molecule_ch1'])

100%|██████████| 6/6 [00:17<00:00,  2.94s/it]


In [3]:
meta_meta

,title,characteristics_ch1,source_name_ch1,library_source,library_strategy,molecule_ch1
GSM1076108,LSEC,"cell type: Liver sinusoidal endothelial cells,...",LSEC,transcriptomic,RNA-Seq,total RNA
GSM1101972,Liver,tissue: liver,Biochain:R-1234149-P,transcriptomic,RNA-Seq,total RNA
GSM1206234,A replicate 1,sample composition: 10 human cell lines (B lym...,Strategene Universal Human Reference RNA (UHRR...,transcriptomic,RNA-Seq,total RNA
GSM1206236,A replicate 2,sample composition: 10 human cell lines (B lym...,Strategene Universal Human Reference RNA (UHRR...,transcriptomic,RNA-Seq,total RNA
GSM1206238,A replicate 3,sample composition: 10 human cell lines (B lym...,Strategene Universal Human Reference RNA (UHRR...,transcriptomic,RNA-Seq,total RNA
...,...,...,...,...,...,...
GSM9470278,1_LX2_Untreated,"tissue: Liver,cell line: LX2,cell type: Epithe...",Liver,transcriptomic,RNA-Seq,total RNA
GSM9470279,2_LX2_GDF15 treated,"tissue: Liver,cell line: LX2,cell type: Epithe...",Liver,transcriptomic,RNA-Seq,total RNA
GSM9470280,2_LX2_Untreated,"tissue: Liver,cell line: LX2,cell type: Epithe...",Liver,transcriptomic,RNA-Seq,total RNA
GSM9470281,3_LX2_GDF15 treated,"tissue: Liver,cell line: LX2,cell type: Epithe...",Liver,transcriptomic,RNA-Seq,total RNA


In [4]:
#filter on characterristics column as consistently the most extensive metadata 

#take only samples that collected all RNA
meta_subset = meta_meta[meta_meta.molecule_ch1 == 'total RNA']
#remove cell lines
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('cell line')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('cell lines')]
#remove cancer and diseased samples
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('cancer')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('disease')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('tumor')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('endothelial')]
meta_subset = meta_subset[~meta_subset.characteristics_ch1.str.contains('developmental')]
#meta_subset = meta_subset[meta_subset['characteristics_ch1'].str.contains('normal|healthy|Healthy')]

In [5]:
meta_subset

,title,characteristics_ch1,source_name_ch1,library_source,library_strategy,molecule_ch1
GSM1101972,Liver,tissue: liver,Biochain:R-1234149-P,transcriptomic,RNA-Seq,total RNA
GSM1377536,R-PHH-DMSO,tissue: Liver,primary human hepatocytes,transcriptomic,RNA-Seq,total RNA
GSM1377537,R-PHH-GW4064,tissue: Liver,primary human hepatocytes,transcriptomic,RNA-Seq,total RNA
GSM1416804,Normal_huLiver,"tissue: Liver tissue,injury: none",Normal human liver biopsy,transcriptomic,RNA-Seq,total RNA
GSM1427132,RNAseq_F-shNT_rep1,"tissue: fetal liver,cell type: CD34+ HSPC-deri...",Human primary fetal live proerythroblasts (Pro...,transcriptomic,RNA-Seq,total RNA
...,...,...,...,...,...,...
GSM9279098,"Isolated Human Hepatocytes, HH36, PNPLA3-WT","tissue: Liver,genotype: WT",Liver,transcriptomic,RNA-Seq,total RNA
GSM9279101,"Isolated Human Hepatocytes, 1, PNPLA3-I148M","tissue: Liver,genotype: I148M",Liver,transcriptomic,RNA-Seq,total RNA
GSM9279102,"Isolated Human Hepatocytes, HH002, PNPLA3-I148M","tissue: Liver,genotype: I148M",Liver,transcriptomic,RNA-Seq,total RNA
GSM9279103,"Isolated Human Hepatocytes, HH003, PNPLA3-I148M","tissue: Liver,genotype: I148M",Liver,transcriptomic,RNA-Seq,total RNA


In [6]:
female_subset = meta_subset[meta_subset.characteristics_ch1.str.contains('female')]
male_subset = meta_subset[meta_subset.characteristics_ch1.str.contains('male')]
#all 'female' samples will be in 'male' - keep only males sampels that are not in female samples
male_subset = male_subset[~male_subset.index.isin(list(female_subset.index))]

In [7]:
print(male_subset.shape)
print(female_subset.shape)

(1162, 6)
(253, 6)


In [8]:
female_subset.to_csv('./data/ARCHS4_female_healthy_meta.csv')
male_subset.to_csv('./data/ARCHS4_male_healthy_meta.csv')
meta_subset.to_csv('./data/ARCHS4_all_healthy_meta.csv')

In [9]:
meta_subset_IDs = list(meta_subset.index)

In [10]:
sample_counts = a4.data.samples(file, meta_subset_IDs)

100%|██████████| 10054/10054 [01:35<00:00, 105.82it/s]


In [ ]:
#filters genes that don't have at least 'readThreshold' reads in 'sampleThreshold' proportion of samples
filtered_exp = a4.utils.filter_genes(sample_counts, readThreshold=50, sampleThreshold=0.02, deterministic=True, aggregate=True)
filtered_exp

In [ ]:
#sampleThreshold = 0.00: SHOULD BE ALL GENES. 62548 genes
#RT = 100, sampleThreshold = 0.02: 24,210 genes
#RT = 1000, sampleThreshold = 0.02: 13,116 genes

In [ ]:
agg_exp = a4.utils.aggregate_duplicate_genes(filtered_exp)

In [ ]:
agg_exp = agg_exp.T
agg_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,AAAS,AACS,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
GSM1101972,32,35,17,11998,48,53,26,63,77,344,...,70,2,167,223,190,11,1430,162,1127,1551
GSM1377536,4865,53,33874,51407,45,12,29,209,2331,2261,...,525,444,339,5885,2455,635,6513,22082,6679,3179
GSM1377537,5490,110,18992,34703,47,15,62,334,1821,2566,...,436,365,339,7862,2473,683,5048,31492,5325,2584
GSM1416804,1926,25,1227,7477,15,1,1,12,106,32,...,18,19,16,402,109,5,196,368,163,108
GSM1427132,17,12,0,6,1,2,0,156,1838,1367,...,1167,3264,43,239,845,7,783,2754,1633,1094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM9279098,6743,49,20841,47068,288,19,14,15,227,225,...,89,11,101,705,753,193,1818,224,1172,971
GSM9279101,3644,88,11698,34633,107,15,106,20,271,232,...,71,11,78,493,735,66,1442,280,953,757
GSM9279102,6418,72,17178,50591,74,28,88,21,276,247,...,72,49,65,559,643,97,4346,356,1384,964
GSM9279103,3806,96,12376,96067,66,21,21,14,330,278,...,83,6,48,782,640,50,1773,526,1276,929


In [ ]:
agg_exp = np.log10(agg_exp + 1)

In [ ]:
#agg_exp.T.to_csv('./data/ARCHS4_healthy_log_stricter.tsv', sep = '\t')